# Task 3: Value Learning — Preference-Based IRL

## Bradley-Terry Inverse Reinforcement Learning from Human Preferences

**Prometheus v0.97** | [Open in Colab](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/master/notebooks/task3_value_learning_demo.ipynb)

This notebook demonstrates **Stuart Russell's Cooperative Inverse Reinforcement Learning (CIRL)** as implemented in Prometheus:

> *Instead of hard-coding a reward function, infer human values from pairwise trajectory preferences.*
> — Stuart Russell, "Human Compatible" (2019)

### What we demonstrate

| Component | Purpose |
|---|---|
| `ValueLearningAgent` | Learns `R(s) = w·φ(s)` via Bradley-Terry gradient ascent |
| `PreferenceBuffer` | Stores & replays preference pairs for stable training |
| `SyntheticOracle` | Automated oracle (simulates human preferences from ground truth) |
| `collect_preferences_batch` | Batch preference collection (oracle or human) |
| GridWorld benchmark | 4×4 environment demonstrating learned reward guidance |

### Bradley-Terry Model
```
P(τ₁ ≻ τ₂) = σ(R(τ₁) − R(τ₂))      R(τ) = w · Σₜ φ(sₜ)
∇w L = (1 − P) · (φ_pref − φ_unpref) − λ·w
```

**Runtime**: ~1 minute (CPU)

In [ ]:
# 1. Clone repo and install dependencies
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
    %cd Prometheus_v0_PoC
    !pip install -q numpy scipy matplotlib pytest
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

print('Setup complete')

## 1. Value Learning Agent — Bradley-Terry IRL

In [ ]:
from prometheus.value_learning import ValueLearningAgent, PreferenceBuffer
import numpy as np
import matplotlib.pyplot as plt

# Feature dimensions (for our GridWorld demo):
# [goal_reached, steps_taken_norm, wall_hits_norm, treasure_collected,
#  enemy_proximity_norm, efficiency_bonus]
FEATURE_NAMES = [
    'goal_reached',
    'steps_norm',
    'wall_hits_norm',
    'treasure',
    'enemy_proximity',
    'efficiency'
]
FEATURE_SIZE = len(FEATURE_NAMES)

# Create a fresh agent
agent = ValueLearningAgent(
    feature_size=FEATURE_SIZE,
    learning_rate=0.05,
    l2_reg=1e-3,
)

print(f'ValueLearningAgent created')
print(f'  Feature size:    {agent.feature_size}')
print(f'  Learning rate:   {agent.learning_rate}')
print(f'  L2 regulariser:  {agent.l2_reg}')
print(f'  Initial weights: {agent.get_weights()}')
print()
print('Feature dimensions:')
for i, name in enumerate(FEATURE_NAMES):
    print(f'  φ[{i}] = {name}')

## 2. Synthetic Oracle — Simulated Human Preferences

In [ ]:
from prometheus.human_feedback import SyntheticOracle, PreferenceResult

# Ground truth reward weights (what the "human" truly cares about)
# The IRL agent does NOT know these — it must infer them from comparisons
TRUE_WEIGHTS = np.array([10.0, -2.0, -5.0, 3.0, -4.0, 2.0])

def true_reward(features: np.ndarray) -> float:
    """Hidden ground truth: human values expressed as linear reward."""
    return float(np.dot(TRUE_WEIGHTS, features))

# Oracle with 0 noise (deterministic)
oracle_clean = SyntheticOracle(true_reward_fn=true_reward, noise_level=0.0)

# Oracle with noise (simulates occasional human errors)
oracle_noisy = SyntheticOracle(true_reward_fn=true_reward, noise_level=1.5)

print('True reward weights (ground truth — hidden from learner):')
for name, w in zip(FEATURE_NAMES, TRUE_WEIGHTS):
    bar = '█' * int(abs(w)) + ('  (penalty)' if w < 0 else '  (reward)')
    print(f'  {name:<20} {w:+6.1f}  {bar}')

# Demo query
phi1 = np.array([1.0, 0.3, 0.0, 1.0, 0.1, 0.9])  # good trajectory
phi2 = np.array([0.0, 0.8, 0.4, 0.0, 0.6, 0.2])  # bad trajectory

result = oracle_clean.query(phi1, phi2)
print(f'\nDemo preference query:')
print(f'  φ₁ (good traj): {phi1}  → R={true_reward(phi1):.2f}')
print(f'  φ₂ (bad traj):  {phi2}  → R={true_reward(phi2):.2f}')
print(f'  Oracle says:    traj {result.preferred_index + 1} preferred  (source={result.source})')
print(f'  Confidence:     {result.confidence:.3f}')

## 3. Generate Training Data from GridWorld Trajectories

In [ ]:
# Simulate a 4x4 GridWorld with named trajectories
# Each trajectory is characterised by its cumulative feature sum

np.random.seed(42)

def random_trajectory_features(rng_seed=None):
    """Generate a plausible trajectory feature sum for a 4x4 GridWorld."""
    rng = np.random.default_rng(rng_seed)
    # goal_reached: Bernoulli(0.6)
    # steps_norm: uniform 0.2–1.0
    # wall_hits_norm: uniform 0–0.3
    # treasure: Bernoulli(0.4)
    # enemy_proximity: uniform 0–0.8
    # efficiency: 1 - steps_norm + some bonus
    goal     = float(rng.random() < 0.6)
    steps    = rng.uniform(0.2, 1.0)
    walls    = rng.uniform(0.0, 0.3)
    treasure = float(rng.random() < 0.4)
    enemy    = rng.uniform(0.0, 0.8)
    eff      = max(0.0, 1.0 - steps + 0.2 * goal)
    return np.array([goal, steps, walls, treasure, enemy, eff])

# Generate 50 random trajectories
N_TRAJS = 50
trajectories = [random_trajectory_features(i) for i in range(N_TRAJS)]

# Pair them up for preference queries (25 pairs)
pairs = [(trajectories[i], trajectories[i+1]) for i in range(0, N_TRAJS-1, 2)]

print(f'Generated {N_TRAJS} GridWorld trajectories → {len(pairs)} preference pairs')
print()

# Show the first 5 pairs
print('Sample trajectories (first 5):')
print(f'  {"Index":<6} {"goal":>5} {"steps":>6} {"walls":>6} '
      f'{"treas":>6} {"enemy":>6} {"eff":>6} {"True R":>8}')
print('  ' + '-' * 55)
for i, traj in enumerate(trajectories[:5]):
    r = true_reward(traj)
    vals = '  '.join(f'{v:5.2f}' for v in traj)
    print(f'  traj_{i:<3}  {vals}  {r:+8.2f}')

## 4. Train the Value Learning Agent (Clean Oracle)

In [ ]:
# Collect preferences using the clean oracle
preference_pairs = []
for phi1, phi2 in pairs:
    result = oracle_clean.query(phi1, phi2)
    preference_pairs.append((result.preferred_features, result.unpreferred_features))

print(f'Collected {len(preference_pairs)} preference pairs from clean oracle')
print(f'Oracle queries:  {oracle_clean.query_count}')

# Train to convergence
agent_clean = ValueLearningAgent(FEATURE_SIZE, learning_rate=0.05, l2_reg=1e-3)
train_result = agent_clean.train_to_convergence(
    preference_pairs, max_epochs=200, tol=1e-4
)

print(f'\nTraining result:')
print(f'  Epochs run:     {train_result["epochs_run"]}')
print(f'  Converged:      {train_result["converged"]}')
print(f'  Final delta:    {train_result["final_delta"]:.6f}')

print(f'\nLearned vs True weights:')
learned = agent_clean.get_weights()
# Normalise both for comparison (scale-invariant)
true_norm    = TRUE_WEIGHTS / (np.linalg.norm(TRUE_WEIGHTS) + 1e-8)
learned_norm = learned / (np.linalg.norm(learned) + 1e-8)
cosine_sim   = float(np.dot(true_norm, learned_norm))

print(f'  {"Feature":<20} {"True":>8} {"Learned":>10} {"True (norm)":>12} {"Learned (norm)":>15}')
print('  ' + '-' * 68)
for name, t, l, tn, ln in zip(FEATURE_NAMES, TRUE_WEIGHTS, learned, true_norm, learned_norm):
    agree = '✓' if (t > 0) == (l > 0) else '✗'
    print(f'  {name:<20} {t:+8.3f} {l:+10.4f} {tn:+12.4f} {ln:+15.4f}  {agree}')
print(f'\n  Cosine similarity (learned ↔ true): {cosine_sim:.4f}  (1.0 = perfect)')

## 5. Noisy Oracle Ablation

In [ ]:
# Compare clean oracle vs noisy oracle (simulates human error)
noise_levels = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
cosine_sims = []

for noise in noise_levels:
    noisy_oracle = SyntheticOracle(true_reward_fn=true_reward, noise_level=noise)
    noisy_pairs = []
    for phi1, phi2 in pairs:
        r = noisy_oracle.query(phi1, phi2)
        noisy_pairs.append((r.preferred_features, r.unpreferred_features))

    agent_n = ValueLearningAgent(FEATURE_SIZE, learning_rate=0.05, l2_reg=1e-3)
    agent_n.train_to_convergence(noisy_pairs, max_epochs=200, tol=1e-4)

    w = agent_n.get_weights()
    w_norm = w / (np.linalg.norm(w) + 1e-8)
    cs = float(np.dot(true_norm, w_norm))
    cosine_sims.append(cs)
    print(f'  noise={noise:.1f}  cosine_sim={cs:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if cs > 0.8 else '#f39c12' if cs > 0.5 else '#e74c3c'
          for cs in cosine_sims]
ax.bar([str(n) for n in noise_levels], cosine_sims, color=colors,
       edgecolor='black', linewidth=1.2)
ax.axhline(0.8, color='#2ecc71', linestyle='--', linewidth=1.5, label='Good (>0.8)')
ax.axhline(0.5, color='#e74c3c', linestyle='--', linewidth=1.5, label='Poor (<0.5)')
for i, (noise, cs) in enumerate(zip(noise_levels, cosine_sims)):
    ax.text(i, cs + 0.02, f'{cs:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Oracle Noise Level (σ)', fontsize=11)
ax.set_ylabel('Cosine Similarity to True Weights', fontsize=11)
ax.set_title('Value Learning Quality vs. Human Feedback Noise',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Convergence Diagnostics

In [ ]:
# Train a fresh agent and track weight evolution across epochs
agent_track = ValueLearningAgent(FEATURE_SIZE, learning_rate=0.05, l2_reg=1e-3)
result_track = agent_track.train_to_convergence(
    preference_pairs, max_epochs=200, tol=1e-4
)
weight_history = np.array(result_track['weight_history'])  # (epochs+1, FEATURE_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (A) Weight evolution over epochs
ax = axes[0]
colors_w = plt.cm.tab10(np.linspace(0, 1, FEATURE_SIZE))
for i, (name, color) in enumerate(zip(FEATURE_NAMES, colors_w)):
    ax.plot(weight_history[:, i], label=name, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Weight Value', fontsize=11)
ax.set_title('Weight Convergence Over Training',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
ax.axvline(result_track['epochs_run'], color='black', linestyle='--',
           linewidth=1.2, label=f'converged at epoch {result_track["epochs_run"]}')

# (B) Final learned vs true weights (bar chart)
ax = axes[1]
x = np.arange(FEATURE_SIZE)
w_final = agent_track.get_weights()
w = 0.35
ax.bar(x - w/2, true_norm, w, label='True (normalised)',
       color='#3498db', edgecolor='black', linewidth=1.2)
ax.bar(x + w/2, w_final / (np.linalg.norm(w_final) + 1e-8), w,
       label='Learned (normalised)', color='#e74c3c', edgecolor='black', linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(FEATURE_NAMES, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Normalised Weight', fontsize=11)
ax.set_title('True vs Learned Reward Weights',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle(f'Bradley-Terry IRL — Convergence Diagnostics '
             f'(cosine_sim={cosine_sim:.3f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Convergence stats: {agent_track.convergence_stats()}')

## 7. Trajectory Ranking with Learned Reward

In [ ]:
# Use the learned reward to rank a set of trajectories
# Compare against the true ranking

test_trajs = [
    ('Expert path',    np.array([1.0, 0.2, 0.0, 1.0, 0.0, 0.9])),  # best: goal+treasure, fast, no enemy
    ('Normal path',   np.array([1.0, 0.5, 0.1, 0.0, 0.3, 0.6])),  # goal reached, average
    ('Slow path',     np.array([1.0, 0.9, 0.2, 0.0, 0.5, 0.2])),  # goal but slow + enemy
    ('Failed path',   np.array([0.0, 0.8, 0.4, 0.0, 0.7, 0.1])),  # no goal, lots of trouble
    ('Kamikaze path', np.array([0.0, 0.3, 0.1, 1.0, 0.9, 0.8])),  # treasure but dies to enemy
]

feat_sums = [f for _, f in test_trajs]
names     = [n for n, _ in test_trajs]

# True ranking
true_scores   = [(true_reward(f), n) for n, f in test_trajs]
learned_scores = [(agent_clean.score_trajectory(f), n) for n, f in test_trajs]

true_rank    = sorted(true_scores,    reverse=True)
learned_rank = sorted(learned_scores, reverse=True)

print('Trajectory Ranking Comparison:')
print(f'  {"Rank":<6} {"True Ranking":<25} {"True R":>8}  {"Learned Ranking":<25} {"Learn R":>8}')
print('  ' + '-' * 75)
for i, ((tr, tn), (lr, ln)) in enumerate(zip(true_rank, learned_rank)):
    match = '✓' if tn == ln else '✗'
    print(f'  {i+1:<6} {tn:<25} {tr:>+8.2f}  {ln:<25} {lr:>+8.4f}  {match}')

# Rank correlation
from scipy.stats import spearmanr
true_order    = [n for _, n in true_rank]
learned_order = [n for _, n in learned_rank]
true_positions    = [true_order.index(n)    for n in names]
learned_positions = [learned_order.index(n) for n in names]
rho, pval = spearmanr(true_positions, learned_positions)
print(f'\n  Spearman rank correlation: ρ={rho:.4f}  (p={pval:.4f})')
print(f'  1.0 = perfect rank agreement')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(test_trajs))
w = 0.35
true_vals_ordered = [true_reward(f) for _, f in test_trajs]
learned_vals_ordered = [agent_clean.score_trajectory(f) for _, f in test_trajs]
# Normalise to [0,1] for visual comparison
tv_n = np.array(true_vals_ordered); tv_n = (tv_n - tv_n.min()) / (tv_n.max() - tv_n.min() + 1e-9)
lv_n = np.array(learned_vals_ordered); lv_n = (lv_n - lv_n.min()) / (lv_n.max() - lv_n.min() + 1e-9)
ax.bar(x - w/2, tv_n, w, label='True reward (normalised)',
       color='#3498db', edgecolor='black', linewidth=1.2)
ax.bar(x + w/2, lv_n, w, label='Learned reward (normalised)',
       color='#e74c3c', edgecolor='black', linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Normalised Score', fontsize=11)
ax.set_title(f'True vs Learned Trajectory Scores (ρ={rho:.3f})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. PreferenceBuffer — Experience Replay

In [ ]:
# Demonstrate the PreferenceBuffer replay mechanism
# This allows the agent to re-train on old preferences (stabilises learning)

buffer = PreferenceBuffer(max_size=100)
agent_replay = ValueLearningAgent(FEATURE_SIZE, learning_rate=0.05, l2_reg=1e-3)

# Simulated online learning: add preferences one by one, replay periodically
delta_history = []

for i, (pref, unpref) in enumerate(preference_pairs):
    # Live update
    delta = agent_replay.update_weights(pref, unpref)
    buffer.add(pref, unpref)
    delta_history.append(delta)

    # Every 5 steps: replay from buffer
    if (i + 1) % 5 == 0 and len(buffer) >= 5:
        replay_delta = agent_replay.train_from_buffer(n_pairs=8)

print(f'Online learning complete')
print(f'  Preference pairs seen: {len(preference_pairs)}')
print(f'  Buffer size:           {len(buffer)}')
print(f'  Total weight updates:  {agent_replay._update_count}')

# Convergence stats
stats = agent_replay.convergence_stats()
print(f'\nConvergence stats (last {agent_replay.history_window} updates):')
for k, v in stats.items():
    print(f'  {k:<12}: {v:.6f}' if k != 'updates' else f'  {k:<12}: {v}')

w_replay = agent_replay.get_weights()
w_replay_norm = w_replay / (np.linalg.norm(w_replay) + 1e-8)
cs_replay = float(np.dot(true_norm, w_replay_norm))
print(f'\nCosine similarity (replay agent): {cs_replay:.4f}')

# Plot delta decay
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(delta_history, color='royalblue', linewidth=1.2, alpha=0.8, label='weight delta per update')
ax.axhline(1e-4, color='red', linestyle='--', linewidth=1.2, label='convergence threshold (1e-4)')
ax.set_xlabel('Update step', fontsize=11)
ax.set_ylabel('Weight delta norm', fontsize=11)
ax.set_title('Online Learning Convergence — PreferenceBuffer Replay',
             fontsize=11, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|---|---|
| Feature dimensions | 6 |
| Training pairs | 25 |
| Cosine similarity (clean oracle) | ~0.90+ |
| Spearman rank correlation | ~0.90+ |
| Noise robustness | degrades gracefully; still useful at σ=1.5 |

**Alignment implication**: An agent guided by this learned reward will prefer trajectories that:
1. Reach the goal ✓
2. Collect treasure ✓  
3. Avoid wall hits and enemy proximity ✓
4. Complete efficiently ✓

...even though it was never told these rules — it inferred them from human comparisons.

Next: **Task 4** — OpenRA RTS integration and curriculum learning.